In [11]:
from typing import Any, Callable, Optional, Union

from pprint import pprint
from datetime import datetime
from pathlib import Path
import os

from torch.utils.tensorboard import SummaryWriter
import pandas as pd
import pytz
import numpy as np
import safetensors.torch as safetensors
import tqdm.notebook as tqdm
import torch
import torch.utils.data as torchdata
import torch.nn as nn
import torchmetrics
import yaml

from flatiron.core.dataset import Dataset
from flatiron.core.types import Compiled, Filepath, Getter
from flatiron.core.tools import get_tensorboard_project
from flatiron.torch.tools import ModelCheckpoint, get_callbacks, TorchDataset, _execute_epoch

Filepath = Union[str, Path]

# QUADRO P6000 is CUDA 6.1
# Triton is CUDA 7.0+
# This tells triton to shutup
import torch._dynamo
torch._dynamo.config.suppress_errors = True

In [12]:
def train(
    device,      # type: str
    model,       # type: torch.nn.Module
    optimizer,   # type: torch.optim.Optimizer
    loss,        # type: torch.nn.Module
    metrics,     # type: list[torch.nn.Module]
    callbacks,   # type: Callbacks
    train_data,  # type: Dataset
    test_data,   # type: Dataset
    params,      # type: dict
    
):
    # type: (...) -> None
    '''
    Train Torch model.

    Args:
        device (str): Device to compile to.
        model (torch.nn.Module): Model to be compiled.
        optimizer (dict): Optimizer config for compilation.
        loss (str): Loss to be compiled.
        metrics (list[str]): Metrics function to be compiled.
        callbacks (dict): Dict of callbacks.
        train_data (Dataset): Training dataset.
        test_data (Dataset): Test dataset.
        params (dict): Training params.
    '''
    checkpoint = callbacks['checkpoint']  # type: Any
    writer = callbacks['tensorboard']
    batch_size = params['batch_size']

    device = torch.device(device)
    torch.manual_seed(params['seed'])
    model = model.to(device)
    loss = loss.to(device)
    metrics = [x.to(device) for x in metrics]

    train_loader = torchdata.DataLoader(
        TorchDataset.monkey_patch(train_data), batch_size=batch_size
    )  # type: torchdata.DataLoader
    test_loader = torchdata.DataLoader(
        TorchDataset.monkey_patch(test_data), batch_size=batch_size
    )  # type: torchdata.DataLoader

    kwargs = dict(
        model=model,
        optimizer=optimizer,
        loss_func=loss,
        device=device,
        metrics_funcs=metrics,
        writer=writer,
    )
    for i in tqdm.trange(params['epochs']):
        _execute_epoch(
            epoch=i, mode='train', data_loader=train_loader,
            checkpoint=checkpoint, **kwargs
        )
        _execute_epoch(epoch=i, mode='test', data_loader=test_loader, **kwargs)
        if checkpoint.save_freq == 'epoch':
            checkpoint.save(model, i)

In [13]:
# DATA
data_kwargs = dict(
    directory='/mnt/storage/projects/unet001/dset002/p-unet001_s-dset002_d-glom_v001',
    label_axis=-1,
    labels=['a'],
)
data = Dataset.read_directory(**data_kwargs)
train_data, test_data = data.train_test_split()
print('DATA')
pprint(data_kwargs)

DATA
{'directory': '/mnt/storage/projects/unet001/dset002/p-unet001_s-dset002_d-glom_v001',
 'label_axis': -1,
 'labels': ['a']}


In [14]:
# MODEL ARCHITECTURE
class Model(nn.Module):
    def __init__(self, input_channels, output_channels):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Conv2d(
                in_channels=input_channels, out_channels=output_channels,
                kernel_size=(3, 3), dtype=torch.float16, padding=1
            ),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.layer_stack(x)

In [92]:
# MODEL
class Conv2DBlock(nn.Module):
    def __init__(self, in_channels, filters=16, dtype=torch.float16):
        super().__init__()
        kwargs = dict(
            out_channels=filters, kernel_size=(3, 3),
            stride=(1, 1), padding=1, padding_mode='reflect', dtype=dtype
        )
        self.conv_1 = nn.Conv2d(in_channels=in_channels, **kwargs)
        self.act_1 = nn.ReLU()
        self.batch_1 = nn.BatchNorm2d(filters, dtype=dtype)
        self.act_1 = nn.Sigmoid()
        self.conv_2 = nn.Conv2d(in_channels=filters, **kwargs)
        self.act_2 = nn.ReLU()
        self.batch_2 = nn.BatchNorm2d(filters, dtype=dtype)

    def forward(self, x):
        x = self.conv_1(x)
        x = self.act_1(x)
        x = self.batch_1(x)
        x = self.conv_2(x)
        x = self.act_2(x)
        x = self.batch_2(x)
        return x


class AtttentionGate2DBlock(nn.Module):
    def __init__(self, in_channels, filters=16, dtype=torch.float16):
        super().__init__()
        kwargs = dict(
            kernel_size=(3, 3),
            stride=(1, 1), padding=1, padding_mode='reflect', dtype=dtype
        )
        self.conv_0 = nn.Conv2d(in_channels=in_channels, out_channels=filters, **kwargs)
        self.conv_1 = nn.Conv2d(in_channels=in_channels, out_channels=filters, **kwargs)
        self.act_1 = nn.ReLU()
        self.conv_2 = nn.Conv2d(in_channels=filters, out_channels=1, **kwargs)
        self.act_1 = nn.Sigmoid()

    def forward(self, skip_connection, query):
        skip = self.conv_0(skip_connection)
        query = self.conv_1(query)

        gate = torch.add(skip, query)
        gate = self.act_1(gate)
        gate = self.conv_2(gate)
        gate = self.act_2(gate)
        gate = torch.multiply(skip, gate)

        x = torch.concatenate([gate, query])
        return x


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, attention=False, dtype=torch.float16):
        super().__init__()
        self._attention = attention
        kwargs = dict(dtype=dtype)
        io_kwargs = dict(kernel_size=(1, 1), stride=(1, 1), dtype=dtype)
        pool_kwargs = dict(kernel_size=(2, 2), stride=(2, 2))
        trans_kwargs = dict(kernel_size=(2, 2), stride=(2, 2), dtype=dtype)
        
        # self.input = nn.Conv2d(in_channels=in_channels, out_channels=16, **io_kwargs)

        self.encode_block_00 = Conv2DBlock(in_channels=in_channels, filters=16, **kwargs)
        self.downsample_00   = nn.MaxPool2d(**pool_kwargs)
        self.encode_block_01 = Conv2DBlock(in_channels=16, filters=32, **kwargs)
        self.downsample_01   = nn.MaxPool2d(**pool_kwargs)

        self.middle_block    = Conv2DBlock(in_channels=32, filters=64, **kwargs)

        self.upsample_00     = nn.ConvTranspose2d(in_channels=64, out_channels=32, **trans_kwargs)  # concat with encode_block_01
        self.decode_block_00 = Conv2DBlock(in_channels=64, filters=32, **kwargs)
        self.upsample_01     = nn.ConvTranspose2d(in_channels=32, out_channels=16, **trans_kwargs)  # concat with encode_block_00
        self.decode_block_01 = Conv2DBlock(in_channels=32, filters=out_channels, **kwargs)

        # self.output = nn.Conv2d(in_channels=16, out_channels=out_channels, **io_kwargs)
        
        # if attention:
        #     self.decode_atten_4 = AtttentionGate2DBlock(in_channels=256, filters=256, **kwargs)
        #     self.decode_atten_3 = AtttentionGate2DBlock(in_channels=256, filters=128, **kwargs)
        #     self.decode_atten_2 = AtttentionGate2DBlock(in_channels=128, filters=64, **kwargs)
        #     self.decode_atten_1 = AtttentionGate2DBlock(in_channels=64, filters=32, **kwargs)

    def forward(self, x):
        # x = self.input(x)
        x0 = self.encode_block_00(x)
        x = self.downsample_00(x0)
        x1 = self.encode_block_01(x)
        x = self.downsample_01(x1)
        x = self.middle_block(x)

        x = self.upsample_00(x)
        x = torch.concatenate([x, x1], axis=1)
        x = self.decode_block_00(x)
        x = self.upsample_01(x)
        x = torch.concatenate([x, x0], axis=1)
        x = self.decode_block_01(x)
        # x = self.output(x)
        return x

    
model_kwargs = dict(
    in_channels=3,
    out_channels=1,
)
model = UNet(**model_kwargs)

# x = torch.rand((1, 3, 208, 208), dtype=torch.float16)
# model(x).shape
print('MODEL')
pprint(model_kwargs)

MODEL
{'in_channels': 3, 'out_channels': 1}


In [94]:
# CALLBACKS
tb = get_tensorboard_project(
    project='unet001',
    root='/mnt/storage/projects',
    extension='safetensors',
)
print('TENSORBOARD')
pprint(tb)

callback_kwargs = dict(
    log_directory=tb['log_dir'],
    checkpoint_pattern=tb['checkpoint_pattern'],
    checkpoint_params=dict(save_freq='epoch'),
)
callbacks = get_callbacks(**callback_kwargs)
print()
print('CALLBACKS')
pprint(callback_kwargs)

# TRAIN KWARGS
train_kwargs = dict(
    device='cuda',
    model=torch.compile(model),
    optimizer=torch.optim.SGD(
        model.parameters(),
        lr=0.001,
    ),
    loss=torchmetrics.JaccardIndex(task='binary', threshold=0.5),
    metrics=[
        torchmetrics.Dice(num_classes=1, threshold=0.5),
    ],
    callbacks=callbacks,
    train_data=train_data,
    test_data=test_data,
    params=dict(
        epochs=10,
        seed=42,
        batch_size=1,
    )
)
print()
print('TRAIN')
pprint(train_kwargs)

# TRAIN
print()
train(**train_kwargs)

TENSORBOARD
{'checkpoint_pattern': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-22-04-54/models/p-unet001_d-2025-02-26_t-22-04-54_e-{epoch:03d}.safetensors',
 'log_dir': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-22-04-54',
 'model_dir': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-22-04-54/models',
 'root_dir': '/mnt/storage/projects/unet001/tensorboard'}

CALLBACKS
{'checkpoint_params': {'save_freq': 'epoch'},
 'checkpoint_pattern': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-22-04-54/models/p-unet001_d-2025-02-26_t-22-04-54_e-{epoch:03d}.safetensors',
 'log_directory': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-22-04-54'}

TRAIN
{'callbacks': {'checkpoint': <flatiron.torch.tools.ModelCheckpoint object at 0x7fbb8862dba0>,
               'tensorboard': <torch.utils.tensorboard.writer.SummaryWriter object at 0x7fbce0ab02b0>},
 'device': 'cuda',
 'loss': BinaryJaccardIndex(),
 'metrics': [Dice()],
 'model': OptimizedM

  0%|          | 0/10 [00:00<?, ?it/s]

RuntimeError: Detected the following values in `target`: tensor([0.0000, 0.0029, 0.0048, 0.0050, 0.0063, 0.0065, 0.0112, 0.0170, 0.0186,
        0.0189, 0.0213, 0.0272, 0.0307, 0.0319, 0.0341, 0.0428, 0.0542, 0.0545,
        0.0561, 0.0586, 0.0695, 0.0722, 0.0731, 0.0739, 0.0749, 0.0767, 0.0840,
        0.0886, 0.1035, 0.1048, 0.1063, 0.1306, 0.1321, 0.1338, 0.1343, 0.1409,
        0.1447, 0.1450, 0.1503, 0.1510, 0.1550, 0.1747, 0.1813, 0.1903, 0.1921,
        0.2041, 0.2142, 0.2279, 0.2612, 0.2642, 0.2830, 0.2854, 0.2900, 0.2935,
        0.3013, 0.3027, 0.3062, 0.3293, 0.3308, 0.3350, 0.3403, 0.3618, 0.3645,
        0.3723, 0.3850, 0.3901, 0.3945, 0.4065, 0.4272, 0.4280, 0.4312, 0.4683,
        0.4719, 0.4768, 0.4771, 0.4785, 0.4795, 0.5005, 0.5010, 0.5034, 0.5073,
        0.5283, 0.5327, 0.5454, 0.5503, 0.5576, 0.5601, 0.5620, 0.5630, 0.5635,
        0.5645, 0.5776, 0.5781, 0.5864, 0.5879, 0.5977, 0.6084, 0.6118, 0.6191,
        0.6201, 0.6240, 0.6343, 0.6353, 0.6372, 0.6387, 0.6479,

In [17]:
!exa --tree /mnt/storage/projects/unet001/tensorboard/d-2025*

/mnt/storage/projects/unet001/tensorboard/d-2025-02-25_t-13-44-28
├── events.out.tfevents.1740509068.5abe6464f7f1.258531.1
└── models
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-000.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-001.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-002.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-003.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-004.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-005.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-006.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-007.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-008.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-009.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-010.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-011.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-012.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-013.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28